# 🎤 Live Egyptian Audio Pipeline Demo

This notebook allows you to test the full pipeline interactively:
1. **Record** your own Egyptian Arabic slang via microphone.
2. **Process** the audio through our Adaptive Preprocessing engine.
3. **Transcribe** using Whisper v3 Large.
4. **Compare** the results before and after cleaning.

In [1]:
# NOTE: Autoreload is disabled to prevent SpeechBrain import loops on Windows
# %load_ext autoreload
# %autoreload 2

import os
import sys
import numpy as np
import IPython.display as ipd
from IPython.display import Javascript
import base64
import librosa
import soundfile as sf
from io import BytesIO

# Add current directory to path
sys.path.append(os.path.abspath('.'))

from pipeline import AdaptivePipeline
from modules.asr import WhisperASR
from utils.audio_io import AudioIO

# Initialize Pipeline and ASR
pipeline = AdaptivePipeline()
asr = WhisperASR()

print("✅ Pipeline and Whisper Model Initialized.")

Initializing SpeechBrain Enhancer on cuda...


c:\Users\Mahmoud\anaconda3\Lib\inspect.py:992: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  if ismodule(module) and hasattr(module, '__file__'):
c:\Users\Mahmoud\anaconda3\Lib\inspect.py:992: UserWarning: Module 'speechbrain.k2_integration' was deprecated, redirecting to 'speechbrain.integrations.k2_fsa'. Please update your script.
  if ismodule(module) and hasattr(module, '__file__'):
c:\Users\Mahmoud\anaconda3\Lib\inspect.py:992: UserWarning: Module 'speechbrain.wordemb' was deprecated, redirecting to 'speechbrain.integrations.huggingface.wordemb'. Please update your script.
  if ismodule(module) and hasattr(module, '__file__'):
c:\Users\Mahmoud\anaconda3\Lib\inspect.py:992: UserWarning: Module 'speechbrain.lobes.models.huggingface_transformers' was deprecated, redirecting to 'speechbrain.integr

✅ SpeechBrain MetricGAN+ Sandboxed & Ready.
✅ Silero VAD loaded successfully.
✅ Pipeline and Whisper Model Initialized.


Using cache found in C:\Users\Mahmoud/.cache\torch\hub\snakers4_silero-vad_master


## 1. Record your Microphone
Run the cell below to start the recorder. Speak some Egyptian slang (e.g., "عامل ايه يا صاحبي، طمنا عليك").

> **Note:** If the browser recorder doesn't show up in VS Code, ensure you have `sounddevice` installed for the fallback system recorder.

In [ ]:
def record_audio(sec=5):
    print(f"🎤 Recording for {sec} seconds... Speak now!")
    
    # Fallback to system recording (Reliable in VS Code Desktop)
    try:
        import sounddevice as sd
        fs = 16000
        recording = sd.rec(int(sec * fs), samplerate=fs, channels=1)
        sd.wait()
        sf.write('live_record.wav', recording, fs)
        print("✅ Recording Saved as live_record.wav")
        return 'live_record.wav'
    except ImportError:
        print("⚠️ 'sounddevice' not found. Please run: pip install sounddevice")
        return None
    except Exception as e:
        print(f"❌ Could not record: {e}")
        return None

live_file = record_audio(sec=5)

## 2. Process & Transcribe
This cell runs the **Adaptive Pipeline** first, then sends the clean audio to **Whisper**.
It also demonstrates the **Confidence Loop**.

In [2]:
# 🔁 Load previous recording instead of recording again

import os

def load_previous_audio(path=None):
    """
    Load an existing audio file instead of recording.
    """
    default_path = "live_record.wav"
    
    if path is None:
        path = default_path
    
    if os.path.exists(path):
        print(f"✅ Loaded existing file: {path}")
        return path
    else:
        print(f"❌ File not found: {path}")
        return None

# 👉 Option 1: reuse last recording automatically
live_file = load_previous_audio()

# 👉 Option 2: specify a file manually
# live_file = load_previous_audio("my_old_recording.wav")

✅ Loaded existing file: live_record.wav


In [3]:
if live_file and os.path.exists(live_file):
    # 1. Original ASR (for comparison)
    orig_audio, sr = AudioIO.load(live_file)
    print("--- 🟢 Original Audio Results ---")
    orig_res = asr.transcribe(orig_audio)
    print(f"Text: {orig_res['text']}")
    print(f"Confidence: {orig_res['confidence']}")
    
    # 2. Adaptive Preprocessing
    print("\n--- 🛠 Applying Adaptive Preprocessing ---")
    proc_res = pipeline.process_file(live_file)
    
    if proc_res['status'] == 'success':
        # Use the array returned by the pipeline directly
        clean_audio = proc_res['processed_audio']
        
        # 3. Processed ASR
        print("--- ✨ Processed Audio Results ---")
        clean_asr_res = asr.transcribe(clean_audio)
        print(f"Text: {clean_asr_res['text']}")
        print(f"Confidence: {clean_asr_res['confidence']}")
        
        # 4. Confidence Loop Logic
        # If Whisper confidence is still too low, we can try even more aggressive denoising
        CONF_THRESHOLD = 0.7
        final_audio = clean_audio
        
        if clean_asr_res['confidence'] < CONF_THRESHOLD:
            print(f"\n⚠️ Confidence {clean_asr_res['confidence']} < {CONF_THRESHOLD}. Retrying with Aggressive settings...")
            # Manual override for aggressive denoising
            retry_audio = pipeline.denoizer.denoise(orig_audio, sr, reduction_factor=0.6)
            retry_res = asr.transcribe(retry_audio)
            print(f"Retry Text: {retry_res['text']}")
            print(f"Retry Confidence: {retry_res['confidence']}")
            if retry_res['confidence'] > clean_asr_res['confidence']:
                final_audio = retry_audio
            
        # 5. Visual/Audio Comparison
        print("\n--- 🎧 Final Comparison ---")
        print("Original Audio:")
        ipd.display(ipd.Audio(orig_audio, rate=sr))
        print("Processed Audio (Cleaned):")
        ipd.display(ipd.Audio(final_audio, rate=sr))
    else:
        print(f"Processing Failed: {proc_res['report']['errors']}")
else:
    print("No recording found. Please run the recording cell above.")

c:\Users\Mahmoud\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
c:\Users\Mahmoud\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
c:\Users\Mahmoud\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


--- 🟢 Original Audio Results ---
Loading Whisper large-v3...
Text: يعني هو ازاي الواحد يقدر يعيش في الدنيا دي كده? وهو
Confidence: 0.796

--- 🛠 Applying Adaptive Preprocessing ---
--- ✨ Processed Audio Results ---
Text: يعني هو ازاي الواحد يقدر يعيش في الدنيا دي كده وهو
Confidence: 0.86

--- 🎧 Final Comparison ---
Original Audio:


Processed Audio (Cleaned):
